In [1]:
!pip install transformers timm einops huggingface_hub -q
!pip install scikit-learn matplotlib seaborn -q

print("Kurulum tamamlandı")

Kurulum tamamlandı


In [2]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
print("HuggingFace login OK")

HuggingFace login OK


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/PRISM'
os.makedirs(f'{BASE_DIR}/embeddings', exist_ok=True)
os.makedirs(f'{BASE_DIR}/results', exist_ok=True)
os.makedirs(f'{BASE_DIR}/figures', exist_ok=True)
print(f"Directories ready: {BASE_DIR}")

Mounted at /content/drive
Dizinler oluşturuldu: /content/drive/MyDrive/PRISM


In [4]:
from transformers import CLIPModel, CLIPProcessor
import torch

model_name = "vinid/plip"
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name)
model = model.cuda()
model.eval()
print("PLIP loaded!")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/568 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: vinid/plip
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PLIP loaded!
GPU memory used: 0.61 GB


In [6]:
import torchvision
import os

os.makedirs('/content/pcam', exist_ok=True)

# torchvision PCam'i otomatik indirir - orijinal dataset
dataset_train = torchvision.datasets.PCAM(
    root='/content/pcam',
    split='train',
    download=True
)

dataset_valid = torchvision.datasets.PCAM(
    root='/content/pcam',
    split='val',
    download=True
)

dataset_test = torchvision.datasets.PCAM(
    root='/content/pcam',
    split='test',
    download=True
)

print(f"Train: {len(dataset_train)} samples")
print(f"Valid: {len(dataset_valid)} samples")
print(f"Test: {len(dataset_test)} samples")

Downloading...
From (original): https://drive.google.com/uc?id=1Ka0XfEMiwgCYPdTI-vv6eUElOBnKFKQ2
From (redirected): https://drive.usercontent.google.com/download?id=1Ka0XfEMiwgCYPdTI-vv6eUElOBnKFKQ2&confirm=t&uuid=98625e5e-9f4a-49e7-84be-63efcb4040c4
To: /content/pcam/pcam/camelyonpatch_level_2_split_train_x.h5.gz
100%|██████████| 6.42G/6.42G [03:00<00:00, 35.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1269yhu3pZDP8UYFQs-NYs3FPwuK-nGSG
To: /content/pcam/pcam/camelyonpatch_level_2_split_train_y.h5.gz
100%|██████████| 21.4k/21.4k [00:00<00:00, 39.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1hgshYGWK8V-eGRy8LToWJJgDU_rXWVJ3
From (redirected): https://drive.usercontent.google.com/download?id=1hgshYGWK8V-eGRy8LToWJJgDU_rXWVJ3&confirm=t&uuid=b9ed9a6e-35d9-417c-bd4c-e69311df5c8a
To: /content/pcam/pcam/camelyonpatch_level_2_split_valid_x.h5.gz
100%|██████████| 806M/806M [00:18<00:00, 43.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bH8ZRbh

Train: 262144 samples
Valid: 32768 samples
Test: 32768 samples


In [7]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms

# PCam için transform - PLIP 224x224 istiyor
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Dataset'leri yeniden yükle (transform ile)
from torchvision.datasets import PCAM

train_dataset = PCAM(root='/content/pcam', split='train', transform=transform)
val_dataset   = PCAM(root='/content/pcam', split='val',   transform=transform)
test_dataset  = PCAM(root='/content/pcam', split='test',  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=False, num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=512, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=512, shuffle=False, num_workers=4)

print(f"Train: {len(train_dataset)}")
print(f"Val:   {len(val_dataset)}")
print(f"Test:  {len(test_dataset)}")

Train: 262144
Val:   32768
Test:  32768


In [9]:
import torch
import numpy as np
from tqdm import tqdm

def extract_features(model, loader, device='cuda'):
    all_features = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = images.to(device)

            # PLIP vision encoder - pooler_output kullan
            outputs = model.vision_model(pixel_values=images)
            features = outputs.pooler_output  # [batch, 512]
            features = features / features.norm(dim=-1, keepdim=True)

            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.concatenate(all_features), np.concatenate(all_labels)

print("Train features çıkarılıyor...")
train_features, train_labels = extract_features(model, train_loader)
print(f"Train: {train_features.shape}")

print("Val features çıkarılıyor...")
val_features, val_labels = extract_features(model, val_loader)
print(f"Val: {val_features.shape}")

print("Test features çıkarılıyor...")
test_features, test_labels = extract_features(model, test_loader)
print(f"Test: {test_features.shape}")

Train features çıkarılıyor...


100%|██████████| 512/512 [03:37<00:00,  2.36it/s]


Train: (262144, 768)
Val features çıkarılıyor...


100%|██████████| 64/64 [00:29<00:00,  2.16it/s]


Val: (32768, 768)
Test features çıkarılıyor...


100%|██████████| 64/64 [00:29<00:00,  2.15it/s]

Test: (32768, 768)


In [10]:
import os
SAVE_DIR = '/content/drive/MyDrive/PRISM/embeddings/plip/pcam'
os.makedirs(SAVE_DIR, exist_ok=True)

np.save(f'{SAVE_DIR}/train_features.npy', train_features)
np.save(f'{SAVE_DIR}/train_labels.npy', train_labels)
np.save(f'{SAVE_DIR}/val_features.npy', val_features)
np.save(f'{SAVE_DIR}/val_labels.npy', val_labels)
np.save(f'{SAVE_DIR}/test_features.npy', test_features)
np.save(f'{SAVE_DIR}/test_labels.npy', test_labels)

print(f"Saved: {SAVE_DIR}")
print(f"Train: {train_features.shape}, {train_labels.shape}")
print(f"Val:   {val_features.shape}, {val_labels.shape}")
print(f"Test:  {test_features.shape}, {test_labels.shape}")

Kaydedildi: /content/drive/MyDrive/PRISM/embeddings/plip/pcam
Train: (262144, 768), (262144,)
Val:   (32768, 768), (32768,)
Test:  (32768, 768), (32768,)


In [11]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
import pandas as pd

def compute_ece(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() > 0:
            acc = y_true[mask].mean()
            conf = y_prob[mask].mean()
            ece += mask.sum() * abs(acc - conf)
    return ece / len(y_true)

def run_label_fraction(train_features, train_labels,
                       val_features, val_labels,
                       test_features, test_labels,
                       fraction, seed=42):

    np.random.seed(seed)
    n = int(len(train_labels) * fraction)
    idx = np.random.choice(len(train_labels), n, replace=False)

    X_train = train_features[idx]
    y_train = train_labels[idx]

    # Linear probe
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(X_train, y_train)

    # Test predictions
    y_prob = clf.predict_proba(test_features)[:, 1]
    y_pred = clf.predict(test_features)

    results = {
        'fraction': fraction,
        'n_samples': n,
        'seed': seed,
        'auroc': roc_auc_score(test_labels, y_prob),
        'f1': f1_score(test_labels, y_pred),
        'brier': brier_score_loss(test_labels, y_prob),
        'ece': compute_ece(test_labels, y_prob),
    }
    return results

# 6 label fraction x 3 seed
fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
seeds = [42, 123, 456]
all_results = []

for frac in fractions:
    for seed in seeds:
        print(f"  Fraction: {frac*100:.0f}%, Seed: {seed}")
        res = run_label_fraction(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        res['model'] = 'PLIP'
        res['dataset'] = 'PCam'
        all_results.append(res)
        print(f"    AUROC: {res['auroc']:.4f} | ECE: {res['ece']:.4f} | Brier: {res['brier']:.4f}")

df = pd.DataFrame(all_results)
print("\n--- SUMMARY (mean over seeds) ---")
print(df.groupby('fraction')[['auroc','ece','brier','f1']].mean().round(4))

  Fraction: 1%, Seed: 42
    AUROC: 0.9424 | ECE: 0.0663 | Brier: 0.1003
  Fraction: 1%, Seed: 123
    AUROC: 0.9406 | ECE: 0.0750 | Brier: 0.1040
  Fraction: 1%, Seed: 456
    AUROC: 0.9417 | ECE: 0.0738 | Brier: 0.1027
  Fraction: 5%, Seed: 42
    AUROC: 0.9511 | ECE: 0.0489 | Brier: 0.0905
  Fraction: 5%, Seed: 123
    AUROC: 0.9509 | ECE: 0.0513 | Brier: 0.0914
  Fraction: 5%, Seed: 456
    AUROC: 0.9485 | ECE: 0.0482 | Brier: 0.0933
  Fraction: 10%, Seed: 42
    AUROC: 0.9521 | ECE: 0.0447 | Brier: 0.0895
  Fraction: 10%, Seed: 123
    AUROC: 0.9527 | ECE: 0.0452 | Brier: 0.0890
  Fraction: 10%, Seed: 456
    AUROC: 0.9519 | ECE: 0.0445 | Brier: 0.0898
  Fraction: 25%, Seed: 42
    AUROC: 0.9532 | ECE: 0.0435 | Brier: 0.0886
  Fraction: 25%, Seed: 123
    AUROC: 0.9532 | ECE: 0.0462 | Brier: 0.0892
  Fraction: 25%, Seed: 456
    AUROC: 0.9531 | ECE: 0.0458 | Brier: 0.0893
  Fraction: 50%, Seed: 42
    AUROC: 0.9531 | ECE: 0.0508 | Brier: 0.0900
  Fraction: 50%, Seed: 123
    AUROC

In [12]:
RESULTS_DIR = '/content/drive/MyDrive/PRISM/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
df.to_csv(f'{RESULTS_DIR}/plip_pcam_results.csv', index=False)
print("Saved!")
print(df.to_string())

Kaydedildi!
    fraction  n_samples  seed     auroc        f1     brier       ece model dataset
0       0.01       2621    42  0.942373  0.863370  0.100268  0.066332  PLIP    PCam
1       0.01       2621   123  0.940551  0.853849  0.104034  0.074964  PLIP    PCam
2       0.01       2621   456  0.941690  0.856065  0.102743  0.073839  PLIP    PCam
3       0.05      13107    42  0.951125  0.870887  0.090541  0.048851  PLIP    PCam
4       0.05      13107   123  0.950868  0.868044  0.091397  0.051294  PLIP    PCam
5       0.05      13107   456  0.948544  0.864879  0.093307  0.048152  PLIP    PCam
6       0.10      26214    42  0.952121  0.869487  0.089527  0.044746  PLIP    PCam
7       0.10      26214   123  0.952703  0.870569  0.088996  0.045185  PLIP    PCam
8       0.10      26214   456  0.951905  0.870350  0.089842  0.044486  PLIP    PCam
9       0.25      65536    42  0.953154  0.870858  0.088632  0.043512  PLIP    PCam
10      0.25      65536   123  0.953161  0.869675  0.089242  0.0

In [13]:
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize_scalar

def temperature_scale(logits, temperature):
    return logits / temperature

def find_optimal_temperature(val_logits, val_labels):
    def nll(T):
        scaled = val_logits / T
        # softmax
        exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
        probs = exp_scaled / exp_scaled.sum(axis=1, keepdims=True)
        # NLL
        nll_val = -np.log(probs[np.arange(len(val_labels)), val_labels] + 1e-10).mean()
        return nll_val

    result = minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded')
    return result.x

def run_with_temperature_scaling(train_features, train_labels,
                                  val_features, val_labels,
                                  test_features, test_labels,
                                  fraction, seed=42):
    np.random.seed(seed)
    n = int(len(train_labels) * fraction)
    idx = np.random.choice(len(train_labels), n, replace=False)

    X_train = train_features[idx]
    y_train = train_labels[idx]

    # Linear probe
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(X_train, y_train)

    # Val logits - temperature bulmak icin
    val_logits = clf.decision_function(val_features).reshape(-1, 1)
    val_logits = np.hstack([-val_logits, val_logits])

    # Optimal temperature bul
    T = find_optimal_temperature(val_logits, val_labels)

    # Test - ham
    test_prob_raw = clf.predict_proba(test_features)[:, 1]
    ece_raw = compute_ece(test_labels, test_prob_raw)

    # Test - temperature scaled
    test_logits = clf.decision_function(test_features).reshape(-1, 1)
    test_logits = np.hstack([-test_logits, test_logits])
    scaled_logits = test_logits / T
    exp_s = np.exp(scaled_logits - scaled_logits.max(axis=1, keepdims=True))
    test_prob_scaled = (exp_s / exp_s.sum(axis=1, keepdims=True))[:, 1]
    ece_scaled = compute_ece(test_labels, test_prob_scaled)

    return {
        'fraction': fraction,
        'seed': seed,
        'temperature': T,
        'ece_raw': ece_raw,
        'ece_scaled': ece_scaled,
        'ece_improvement': ece_raw - ece_scaled,
        'auroc': roc_auc_score(test_labels, test_prob_raw),
        'brier_raw': brier_score_loss(test_labels, test_prob_raw),
        'brier_scaled': brier_score_loss(test_labels, test_prob_scaled),
    }

# Run
fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
seeds = [42, 123, 456]
ts_results = []

for frac in fractions:
    for seed in seeds:
        res = run_with_temperature_scaling(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        res['model'] = 'PLIP'
        res['dataset'] = 'PCam'
        ts_results.append(res)

df_ts = pd.DataFrame(ts_results)
print("--- Temperature Scaling Results ---")
summary = df_ts.groupby('fraction')[['temperature','ece_raw','ece_scaled','ece_improvement']].mean().round(4)
print(summary)

--- Temperature Scaling Sonuclari ---
          temperature  ece_raw  ece_scaled  ece_improvement
fraction                                                   
0.01           1.2784   0.0717      0.0334           0.0384
0.05           1.6557   0.0494      0.0386           0.0108
0.10           1.8046   0.0448      0.0400           0.0048
0.25           1.9624   0.0452      0.0454          -0.0003
0.50           2.0479   0.0523      0.0519           0.0004
1.00           2.1417   0.0574      0.0560           0.0014


In [14]:
df_ts.to_csv(f'{RESULTS_DIR}/plip_pcam_temperature_scaling.csv', index=False)
print("Saved!")

Kaydedildi!
